# BankEff_R — Reusable Template

Any institution-year file with a time column, a performance metric (ROA, NIM, NPL, efficiency), a size column, a name, and a charter / product flag.


In [ ]:
library(ggplot2); library(dplyr); library(tidyr); library(readr)
theme_set(theme_minimal())

path <- "data/banks.csv"
time_col <- "year"
y_col <- "roa"
size_col <- "assets_b"
maker_col <- "bank"
fuel_col <- "charter"
keep_fuels <- c("Community Commercial", "Regional Commercial", "Money Center")
exclude_tech <- c("Digital-only")
tech_col <- "atvType"
trans_col <- "funding"
book_window <- c(1990, 2014)


In [ ]:
raw <- read_csv(path, show_col_types = FALSE)
df <- raw %>% rename(.time = !!time_col, .y = !!y_col, .maker = !!maker_col)
if (size_col %in% names(raw)) df$.size <- as.numeric(raw[[size_col]])
if (fuel_col %in% names(raw)) df$.fuel <- raw[[fuel_col]]
if (tech_col %in% names(raw)) df$.tech <- raw[[tech_col]]
if (trans_col %in% names(raw)) df$.trans <- raw[[trans_col]]
cat("rows", nrow(df), "time", min(df$.time), "-", max(df$.time), "\n")


In [ ]:
all_yr <- df %>% group_by(.time) %>% summarise(avg = mean(.y, na.rm = TRUE), series = "All")
core <- df
if (".fuel" %in% names(df)) core <- filter(core, .fuel %in% keep_fuels)
if (".tech" %in% names(df)) core <- filter(core, is.na(.tech) | !(.tech %in% exclude_tech))
core_yr <- core %>% group_by(.time) %>% summarise(avg = mean(.y, na.rm = TRUE), series = "Core")
bind_rows(all_yr, core_yr) %>%
  ggplot(aes(.time, avg, color = series)) + geom_line() + geom_point(size = 1.2) +
  labs(x = "Year", y = "Mean metric", title = "All vs filtered core")


In [ ]:
if (".size" %in% names(df)) {
  print(ggplot(core, aes(log10(pmax(.size, 0.05)), .y)) + geom_point(alpha = 0.15) + geom_smooth() +
          labs(title = "Size vs metric (core)"))
  dual <- core %>% group_by(.time) %>%
    summarise(metric = mean(.y, na.rm = TRUE), size = mean(.size, na.rm = TRUE)) %>% pivot_longer(-.time)
  print(ggplot(dual, aes(.time, value)) + geom_point() + geom_smooth() +
          facet_wrap(~name, ncol = 1, scales = "free_y"))
}


In [ ]:
w <- core %>% filter(.time >= book_window[1], .time <= book_window[2])
common <- Reduce(intersect, lapply(split(w$.maker, w$.time), unique))
print(sort(common))
core %>% filter(.maker %in% common) %>%
  group_by(.time, .maker) %>% summarise(avg = mean(.y, na.rm = TRUE)) %>%
  ggplot(aes(.time, avg)) + geom_line() + facet_wrap(~.maker) +
  labs(title = "Common institutions across the window")


Map columns, edit `keep_fuels`, rewrite the four audience paragraphs. Keep the short `BankXxx_R` prefix.
